# Feature Engineering

## Objective

The objective of this notebook is to transform the cleaned music dataset into a numerical representation suitable for recommendation modeling.

The feature engineering process focuses on:

- Separating features according to their semantic roles
- Selecting features relevant to content-based recommendation
- Applying appropriate transformations to numerical features
- Constructing a consistent feature matrix for similarity-based recommendation
- Preserving track identifiers and metadata separately from the modeling representation

The resulting feature matrix will be used in the subsequent content-based recommendation system.

In [1]:
import pandas as pd
import numpy as np

In [3]:
DATA_PATH = "../data/processed/music_clean.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

df.shape

(157490, 19)

In [5]:
df.head()

,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921.0,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667.0,0.211,0.0,4BJqT0PrAfrxzMOxytFOIz,0.878000,10.0,0.665,-20.096,1.0,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4.0,1921,0.0366,80.954
1,0.9630,1921.0,0.732,['Dennis Day'],0.819,180533.0,0.341,0.0,7xPhfUan2yNtyFG0cUWkt8,0.000000,7.0,0.160,-12.441,1.0,Clancy Lowered the Boom,5.0,1921,0.4150,60.936
2,0.0394,1921.0,0.961,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062.0,0.166,0.0,1o6I8BglA6ylDMrIELygv1,0.913000,3.0,0.101,-14.850,1.0,Gati Bali,5.0,1921,0.0339,110.339
3,0.1650,1921.0,0.967,['Frank Parker'],0.275,210000.0,0.309,0.0,3ftBPsC5vPBKxYSee08FDH,0.000028,5.0,0.381,-9.316,1.0,Danny Boy,3.0,1921,0.0354,100.109
4,0.2530,1921.0,0.957,['Phil Regan'],0.418,166693.0,0.193,0.0,4d6HGyGT8e121BsdKmw9v6,0.000002,3.0,0.229,-10.096,1.0,When Irish Eyes Are Smiling,2.0,1921,0.0380,101.665


In [6]:
audio_features = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "valence",
    "tempo"
]

musical_features = [
    "key",
    "mode"
]

track_features = [
    "duration_ms"
]

context_features = [
    "popularity",
    "explicit"
]

## Feature Groups

The available numerical attributes are separated according to their semantic role rather than treating every numerical column as an equivalent audio feature.

### Core audio features

These describe continuous characteristics of the musical content:

- acousticness
- danceability
- energy
- instrumentalness
- liveness
- loudness
- speechiness
- valence
- tempo

### Musical features

`key` and `mode` describe musical properties but have discrete representations.

### Track-level feature

`duration_ms` represents the duration of a track. Although numerical and continuous, its semantic meaning differs from the core audio characteristics.

### Context features

`popularity` and `explicit` provide additional information about a track but are not direct measurements of its acoustic similarity.

These groups will initially be treated separately so that their contribution to recommendation similarity can be evaluated deliberately rather than implicitly.

In [8]:
content_features = audio_features.copy()

In [9]:
content_features 

['acousticness',
 'danceability',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'valence',
 'tempo']

In [10]:
df[content_features].describe().T

,count,mean,std,min,25%,50%,75%,max
acousticness,157490.0,0.509352,0.377327,0.0,0.10600,0.532000,0.903000,0.996
danceability,157490.0,0.537658,0.176132,0.0,0.41500,0.548000,0.669000,0.988
energy,157490.0,0.477625,0.267541,0.0,0.25000,0.463000,0.698000,1.000
instrumentalness,157490.0,0.169714,0.315752,0.0,0.00000,0.000227,0.110000,1.000
liveness,157490.0,0.205484,0.173417,0.0,0.09900,0.136000,0.261000,0.999
loudness,157490.0,-11.533284,5.711133,-60.0,-14.71600,-10.648000,-7.225000,3.855
speechiness,157490.0,0.100117,0.166722,0.0,0.03500,0.045100,0.076175,0.970
valence,157490.0,0.528954,0.263286,0.0,0.31700,0.540000,0.748000,1.000
tempo,157490.0,116.712327,30.734610,0.0,93.21825,114.579000,135.354750,243.507


In [12]:
df[content_features].isna().sum()

acousticness        0
danceability        0
energy              0
instrumentalness    0
liveness            0
loudness            0
speechiness         0
valence             0
tempo               0
dtype: int64

In [13]:
df[content_features].skew().sort_values(ascending=False)

speechiness         3.972459
liveness            2.147890
instrumentalness    1.605367
tempo               0.449549
energy              0.133830
acousticness       -0.059475
valence            -0.110010
danceability       -0.225117
loudness           -1.026789
dtype: float64

## Feature Distribution Analysis

The skewness of the selected audio features is examined before scaling.

Several features show substantial positive skewness, particularly `speechiness`, `liveness`, and `instrumentalness`. Standardization does not remove this skewness; it only places the features on a comparable scale.

At this stage, no transformation is applied automatically. The effect of potential transformations will be evaluated during model development because transforming a feature can change the geometry of the feature space and therefore affect similarity-based recommendations.

The initial modeling representation will therefore use standardized features as a baseline.

In [14]:
X = df[content_features].copy()

In [16]:
X.shape

(157490, 9)

In [17]:
X.head()

,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,valence,tempo
0,0.982,0.279,0.211,0.878000,0.665,-20.096,0.0366,0.0594,80.954
1,0.732,0.819,0.341,0.000000,0.160,-12.441,0.4150,0.9630,60.936
2,0.961,0.328,0.166,0.913000,0.101,-14.850,0.0339,0.0394,110.339
3,0.967,0.275,0.309,0.000028,0.381,-9.316,0.0354,0.1650,100.109
4,0.957,0.418,0.193,0.000002,0.229,-10.096,0.0380,0.2530,101.665


In [18]:
from sklearn.preprocessing import StandardScaler

In [19]:
scaler = StandardScaler()

In [20]:
X_scaled =  scaler.fit_transform(X)

In [21]:
X_scaled = pd.DataFrame(X_scaled, columns=content_features, index=df.index)

In [22]:
X_scaled.head()

,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,valence,tempo
0,1.252625,-1.468552,-0.996581,2.243176,2.649779,-1.499307,-0.380975,-1.783443,-1.163458
1,0.590068,1.597342,-0.510672,-0.537494,-0.262284,-0.158939,1.888681,1.648580,-1.814778
2,1.196970,-1.190350,-1.164780,2.354023,-0.602505,-0.580748,-0.397170,-1.859407,-0.207367
3,1.212872,-1.491262,-0.630280,-0.537407,1.012104,0.388240,-0.388173,-1.382357,-0.540218
4,1.186370,-0.679368,-1.063860,-0.537489,0.135602,0.251664,-0.372578,-1.048118,-0.489591


In [23]:
X_scaled.describe().T

,count,mean,std,min,25%,50%,75%,max
acousticness,157490.0,-6.352430e-17,1.000003,-1.349899,-1.068975,0.060023,1.043257,1.289728
danceability,157490.0,2.223350e-16,1.000003,-3.052597,-0.696401,0.058717,0.745705,2.556853
energy,157490.0,-6.929923e-17,1.000003,-1.785248,-0.850808,-0.054665,0.823709,1.952513
instrumentalness,157490.0,-2.526535e-17,1.000003,-0.537494,-0.537494,-0.536775,-0.189119,2.629557
liveness,157490.0,1.616982e-16,1.000003,-1.184917,-0.614038,-0.400679,0.320129,4.575777
loudness,157490.0,4.908696e-17,1.000003,-8.486384,-0.557285,0.155011,0.754368,2.694445
speechiness,157490.0,2.887468e-18,1.000003,-0.600503,-0.390572,-0.329992,-0.143603,5.217590
valence,157490.0,-4.619949e-17,1.000003,-2.009054,-0.805036,0.041956,0.831974,1.789112
tempo,157490.0,5.327379e-16,1.000003,-3.797435,-0.764420,-0.069411,0.606563,4.125482


## Scale Continuous Audio Features

The selected audio features have different numerical ranges. For example, `tempo` is measured in beats per minute, while several other features are normalized to approximately the range 0–1.

Standardization is therefore applied so that features with larger numerical scales do not dominate distance or similarity calculations solely because of their units.

StandardScaler transforms each feature approximately according to:

z = (x - μ) / σ

where μ is the feature mean and σ is the feature standard deviation.

The standardized representation will be used as the initial feature space for similarity-based recommendation.

In [25]:
X_scaled.shape

(157490, 9)

In [27]:
X_scaled.isna().sum().sum()

np.int64(0)

In [29]:
np.isfinite(X_scaled).all().all()

np.True_

In [30]:
X_scaled.columns.tolist()

['acousticness',
 'danceability',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'valence',
 'tempo']

In [31]:
X_scaled.to_csv("../data/processed/audio_features_scaled.csv", index=False)

In [32]:
track_metadata = df[
    [
        "id",
        "name",
        "artists",
        "year",
        "release_date",
        "popularity",
        "explicit"
    ]
].copy()

In [34]:
track_metadata.shape

(157490, 7)

In [35]:
track_metadata.head()

,id,name,artists,year,release_date,popularity,explicit
0,4BJqT0PrAfrxzMOxytFOIz,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...","['Sergei Rachmaninoff', 'James Levine', 'Berli...",1921.0,1921,4.0,0.0
1,7xPhfUan2yNtyFG0cUWkt8,Clancy Lowered the Boom,['Dennis Day'],1921.0,1921,5.0,0.0
2,1o6I8BglA6ylDMrIELygv1,Gati Bali,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,1921.0,1921,5.0,0.0
3,3ftBPsC5vPBKxYSee08FDH,Danny Boy,['Frank Parker'],1921.0,1921,3.0,0.0
4,4d6HGyGT8e121BsdKmw9v6,When Irish Eyes Are Smiling,['Phil Regan'],1921.0,1921,2.0,0.0


In [36]:
track_metadata["id"].nunique()

157490

In [33]:
track_metadata.to_csv("../data/processed/track_metadata.csv", index=False)

In [37]:
assert len(X_scaled) == len(track_metadata)
assert X_scaled.index.equals(track_metadata.index)

In [38]:
assert X_scaled.shape[1] == len(content_features)
assert list(X_scaled.columns) == content_features
assert X_scaled.isna().sum().sum() == 0
assert np.isfinite(X_scaled).all().all()